# Clasificación de Letras Escritas a Mano con Redes Convolucionales (CNNs)

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Asignatura Ministerial:** Procesamiento Digital de Imágenes
* **Nombre de Trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

El objetivo de esta sesión es diseñar, entrenar y evaluar una **Red Neuronal Convolucional (CNN)** para clasificar automáticamente imágenes de letras escritas a mano (dataset EMNIST). Estudiaremos la adición de filtros convolucionales 2D y capas de reducción espacial, contrastando directamente la superioridad de esta arquitectura espacial frente al modelo de Perceptrón Multicapa (MLP) analizado en el cuaderno anterior.

## Resultados de aprendizaje

Al final de este notebook van a poder:
1. Explicar el propósito y ventajas de las capas convolucionales 2D en Procesamiento Digital de Imágenes.
2. Diseñar una arquitectura convolucional secuencial uniendo capas `Conv2D`, `MaxPooling2D`, `Flatten` y `Dense`.
3. Entrenar y graficar las curvas de precisión y error de una CNN.
4. Comparar empíricamente y teóricamente el desempeño de un Perceptrón Multicapa (MLP) contra una CNN sobre el mismo conjunto de datos.

## Terminología clave (Microglosario)

*   **Red Convolucional (CNN):** Modelo de aprendizaje profundo diseñado para procesar datos bidimensionales preservando sus relaciones espaciales y de vecindad.
*   **Capa Convolucional (`Conv2D`):** Capa que desplaza un conjunto de filtros (*kernels*) sobre la imagen para extraer mapas de características locales como bordes o contornos.
*   **Capa de Reducción (`MaxPooling2D`):** Operación que reduce el ancho y alto de los mapas de características reteniendo únicamente la activación máxima en pequeñas regiones.
*   **Invarianza Espacial:** Capacidad del modelo de reconocer un patrón o contorno sin importar su posición o ligera traslación dentro de la imagen.

## 1. Configuración de Librerías y Herramientas

Carguemos las herramientas esenciales de TensorFlow, visualización y análisis cuantitativo de datos.

In [ ]:
print("✦ Instalando dependencias en el sistema...")
!pip install opencv-python-headless seaborn -q
print("✓ Librerías instaladas con éxito.\n")

import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import seaborn as sns
from sklearn.metrics import confusion_matrix

print("✓ Entorno deDeep Learning listo para operar.")

## 2. Carga y Preprocesamiento de EMNIST

Para realizar una comparación justa y directa contra el modelo anterior, utilizaremos el mismo dataset de letras escritas a mano (EMNIST/Letters) bajo idénticos parámetros de normalización y estructuración de etiquetas.

In [ ]:
print("✦ Descargando y cargando EMNIST/Letters en memoria...")
datos, metadatos = tfds.load('emnist/letters', as_supervised=True, with_info=True)
print("✓ Dataset cargado correctamente.\n")

# Generamos de forma explícita el vector de etiquetas
nombres_clases = []
for i in range(1, 27):
    letra = chr(i + ord('a') - 1)
    nombres_clases.append(letra)

print("✓ Clases listas:", nombres_clases)

In [ ]:
def preprocesar_imagen(imagen, etiqueta):
    # Convertimos a flotante y normalizamos al rango [0, 1]
    imagen_normalizada = tf.cast(imagen, tf.float32) / 255.0
    # Desplazamos la etiqueta para iniciar en 0
    etiqueta_ajustada = etiqueta - 1
    return imagen_normalizada, etiqueta_ajustada

# Preparamos los conjuntos de datos en caché para máxima velocidad
datos_entrenamiento = datos['train'].map(preprocesar_imagen).cache()
datos_pruebas = datos['test'].map(preprocesar_imagen).cache()

print("✓ Datos preprocesados con éxito.")

## 3. Preparación de Lotes para el Entrenamiento

Agruparemos las muestras en lotes de tamaño 32 y las barajaremos para asegurar un gradiente de optimización equilibrado y libre de sesgos secuenciales.

In [ ]:
TAMANO_LOTE = 32
total_muestras_entrenamiento = metadatos.splits['train'].num_examples

# Dividimos y mezclamos secuencialmente
datos_entrenamiento_lotes = datos_entrenamiento.shuffle(total_muestras_entrenamiento)
datos_entrenamiento_lotes = datos_entrenamiento_lotes.batch(TAMANO_LOTE)
datos_entrenamiento_lotes = datos_entrenamiento_lotes.repeat()

datos_pruebas_lotes = datos_pruebas.batch(TAMANO_LOTE)

print(f"✓ Lotes de {TAMANO_LOTE} configurados correctamente para la CNN.")

## 4. Diseño y Construcción de la Red Neuronal Convolucional (CNN)

A diferencia del Perceptrón Multicapa, la **CNN** recibe y procesa la imagen conservando su formato original 2D de $28 \times 28 \times 1$. Aplicaremos filtros convolucionales alternados con capas de Max-Pooling para extraer y resumir rasgos geométricos abstractos.

In [ ]:
# Construimos la arquitectura de la CNN
modelo_cnn = tf.keras.Sequential([
    # Capa Convolucional 1: 16 filtros de 3x3 y activación ReLU
    tf.keras.layers.Conv2D(16, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    # Capa de MaxPooling 1: reduce la resolución espacial a la mitad
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    
    # Capa Convolucional 2: 32 filtros de 3x3 y activación ReLU
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    # Capa de MaxPooling 2: reduce la resolución espacial a la mitad
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    
    # Aplanado: transformamos los mapas de características resultantes a vector plano
    tf.keras.layers.Flatten(),
    
    # Capa Totalmente Conectada (Dense) intermedia
    tf.keras.layers.Dense(64, activation='relu'),
    
    # Capa de salida: una neurona por letra de la 'a' a la 'z'
    tf.keras.layers.Dense(len(nombres_clases), activation='softmax')
])

# Compilación con optimizador y pérdida multiclase
modelo_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✓ Arquitectura de la CNN diseñada y compilada.")
modelo_cnn.summary()

## Consigna de Lectura e Interpretación

Revisen el bloque de `summary()`. Observen la transición de las dimensiones espaciales de salida a medida que la imagen atraviesa la primera y segunda capa de `Conv2D` y `MaxPooling2D`. ¿Cómo cambia el tamaño del tensor?

## 5. Entrenamiento de la CNN

Procederemos al entrenamiento del modelo por un total de 15 épocas completas, análogo al entrenamiento del MLP anterior.

In [ ]:
EPOCAS = 15
pasos_por_epoca = math.ceil(total_muestras_entrenamiento / TAMANO_LOTE)

print(f"✦ Iniciando entrenamiento de la CNN por {EPOCAS} épocas...")
print("  (Observen la precisión alcanzada desde las primeras épocas)\n")

historial = modelo_cnn.fit(
    datos_entrenamiento_lotes,
    epochs=EPOCAS,
    steps_per_epoch=pasos_por_epoca
)

print("\n✓ Entrenamiento finalizado con éxito.")

## 6. Evaluación de la CNN en Datos de Prueba

Validaremos la exactitud final alcanzada en el conjunto de pruebas que el modelo no procesó durante su optimización.

In [ ]:
print("✦ Evaluando la CNN en datos no vistos...")
perdida_prueba, precision_prueba = modelo_cnn.evaluate(datos_pruebas_lotes, verbose=0)

print(f"\nResultados en Datos de Prueba (CNN):")
print(f"  • Pérdida (Loss) calculada: {perdida_prueba:.4f}")
print(f"  • Precisión (Accuracy) calculada: {precision_prueba:.4f} ({precision_prueba * 100:.2f}%)")

## 7. Contraste y Comparación Analítica: MLP vs. CNN

A continuación, realizaremos una comparación de rendimiento entre ambos clasificadores basándonos en sus arquitecturas matemáticas.

### Cuadro Comparativo Teórico-Empírico

| Métrica / Aspecto | Perceptrón Multicapa (MLP - Cuaderno 02) | Red Convolucional (CNN - Cuaderno 03) |
| :--- | :--- | :--- |
| **Tratamiento Espacial** | Aplanado unidimensional (destruye topología) | Preserva la estructura 2D bidimensional |
| **Extracción de Patrones** | Manual implícita global | Convolución local jerárquica con filtros |
| **Invarianza a Traslaciones** | Extremadamente baja (sensible al desfase) | Muy alta (robusta ante ligeros desplazamientos) |
| **Precisión Promedio (EMNIST)**| ~80% - 84% | **~90% - 94%** |
| **Parámetros del summary()** | Elevados en primera capa | Distribución equilibrada gracias al Max-Pooling |

## Consigna de Lectura e Interpretación

Analicen la tabla anterior y confronten los resultados empíricos que obtuvieron en el laboratorio. ¿A qué se debe el gran incremento en precisión de la CNN por sobre el MLP? ¿De qué forma afecta el hecho de aplanar la imagen en un vector unidimensional al procesamiento de contornos complejos?

## Cierre de Laboratorio

Han comprobado empíricamente la enorme diferencia en rendimiento y robustez espacial que aporta una Red Convolucional (CNN) en Procesamiento Digital de Imágenes en comparación con un Perceptrón clásico.

En la siguiente sesión ingresaremos al núcleo de estas capas de convolución y pooling para visualizar de manera exacta cómo reaccionan las neuronas e inspeccionar el contenido de sus filtros.